# 01 — PoC: Single SIMON Scan → Face Render → FaceAge

**Goal**: Verify the full pipeline works end-to-end on one scan before running at scale.

Steps:
1. Load a SIMON T1 scan
2. Inspect orientation and confirm face is present (6-view plot)
3. Render a frontal 2D face PNG with PyVista
4. Run FaceAge on the render
5. Diagnostic summary

**Setup**: Edit `SCAN_PATH` and `FACEAGE_ROOT` below.

In [ ]:
import sys
from pathlib import Path

# ── CONFIG ──────────────────────────────────────────────────────────────────
SCAN_PATH   = Path("../data/simon/simon_freesurfer8_ses-001_mri_orig.mgz")
FACEAGE_ROOT = Path("../vendor/FaceAge")
OUT_DIR      = Path("../results/poc")
SKIN_LEVEL   = 40   # isosurface threshold — adjust if render looks wrong
BYPASS_MTCNN = True  # recommended for MRI renders
# ────────────────────────────────────────────────────────────────────────────

OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(Path(".").resolve().parent))

print(f"Scan   : {SCAN_PATH}  exists={SCAN_PATH.exists()}")
print(f"FaceAge: {FACEAGE_ROOT}  exists={FACEAGE_ROOT.exists()}")

In [ ]:
# ── 1. Load volume ───────────────────────────────────────────────────────────
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

from src.utils import load_vol, reorient_to_ras

img = reorient_to_ras(load_vol(SCAN_PATH))
data = img.get_fdata(dtype=np.float32)

print(f"Shape  : {data.shape}")
print(f"Dtype  : {data.dtype}")
print(f"Range  : [{data.min():.1f}, {data.max():.1f}]")
print(f"Affine :\n{img.affine}")

In [ ]:
# ── 2. Six-view sanity check (confirm face is present, not defaced) ──────────
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
cx, cy, cz = [s // 2 for s in data.shape]

views = [
    ("Axial (mid)",    data[:, :, cz].T,                  "gray"),
    ("Coronal (mid)",  data[:, cy, :].T,                  "gray"),
    ("Sagittal (mid)", data[cx, :, :].T,                  "gray"),
    # anterior slices — this is where the face should be
    ("Coronal (ant)",  data[:, int(cy * 1.6), :].T,       "gray"),
    ("Axial (sup)",    data[:, :, int(cz * 1.4)].T,       "gray"),
    ("Sagittal (lat)", data[int(cx * 1.2), :, :].T,       "gray"),
]

for ax, (title, sl, cmap) in zip(axes.flat, views):
    ax.imshow(sl, cmap=cmap, origin="lower", vmin=0, vmax=np.percentile(data, 99))
    ax.set_title(title, fontsize=10)
    ax.axis("off")

plt.suptitle(f"{SCAN_PATH.name}  shape={data.shape}", fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "01_volume_slices.png", dpi=120)
plt.show()
print("→ Check the 'Coronal (ant)' slice: the face should be visible there.")

In [ ]:
# ── 3. Render frontal face PNG ───────────────────────────────────────────────
from src.render import render_face

out_png = OUT_DIR / "face_render.png"
ok = render_face(SCAN_PATH, out_png, level=SKIN_LEVEL)

print(f"render_face returned: {ok}")
if ok:
    from PIL import Image
    img_render = Image.open(out_png)
    plt.figure(figsize=(5, 5))
    plt.imshow(img_render)
    plt.axis("off")
    plt.title("Frontal face render")
    plt.show()
else:
    print("⚠ Render failed. Try adjusting SKIN_LEVEL (e.g. 20, 30, 50).")

In [ ]:
# ── 3b. If render looks wrong, try different levels ──────────────────────────
from src.render import render_face

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
levels = [20, 30, 40, 60]

for ax, lv in zip(axes, levels):
    tmp_png = OUT_DIR / f"face_level_{lv}.png"
    ok = render_face(SCAN_PATH, tmp_png, level=lv)
    if ok:
        ax.imshow(plt.imread(tmp_png))
    else:
        ax.text(0.5, 0.5, "failed", ha="center", va="center")
    ax.set_title(f"level={lv}")
    ax.axis("off")

plt.suptitle("Threshold sweep — pick the best one for SKIN_LEVEL")
plt.tight_layout()
plt.savefig(OUT_DIR / "01_level_sweep.png", dpi=120)
plt.show()

In [ ]:
# ── 4. Run FaceAge ───────────────────────────────────────────────────────────
from src.face_age import predict_age

result = predict_age(
    img_path=out_png,
    faceage_root=FACEAGE_ROOT,
    bypass_mtcnn=BYPASS_MTCNN,
)

print("FaceAge result:")
for k, v in result.items():
    if k == "embedding":
        print(f"  embedding : shape={v.shape}  norm={np.linalg.norm(v):.3f}")
    else:
        print(f"  {k:15s}: {v}")

In [ ]:
# ── 5. Summary ───────────────────────────────────────────────────────────────
print("=" * 50)
print("PoC Summary")
print("=" * 50)
print(f"Scan         : {SCAN_PATH.name}")
print(f"Volume shape : {data.shape}")
print(f"Render OK    : {ok}")
print(f"MTCNN found  : {result['mtcnn_found']}")
print(f"Bypass used  : {result['bypass_used']}")
print(f"Face age     : {result['age']:.1f} years" if not np.isnan(result['age']) else "Face age     : NaN (check model weights)")
print("=" * 50)
print()
print("Next steps:")
print("  • If face age is NaN → check vendor/FaceAge/models/ weights are downloaded")
print("  • If render failed → adjust SKIN_LEVEL using the level sweep above")
print("  • If MTCNN found=False → bypass_mtcnn=True is correct for MRI renders")
print("  • Proceed to 02_simon_reliability.ipynb for all 73 sessions")